# Week 89, Day 3: Build

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w089d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark_domain.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\n\n\nclass UnknownPlace(LookupError):\n    pass\n\n\nclass NoRoute(Exception):\n    pass\n\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\nclass Route:\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        return naismith_minutes(self.km, self.climb_m)\n\n    def __repr__(self):\n        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"\n\n\nclass PathNetwork:\n    def __init__(self, heights):\n        self.heights = dict(heights)\n        self.paths = {name: {} for name in self.heights}\n\n    def _check(self, *names):\n        for name in names:\n            if name not in self.heights:\n                raise UnknownPlace(name)\n\n    def add_path(self, a, b, km):\n        self._check(a, b)\n        if km <= 0:\n            raise ValueError("km must be positive")\n        self.paths[a][b] = km\n        self.paths[b][a] = km\n\n    def climb(self, stops):\n        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))\n\n    def shortest(self, start, end):\n        self._check(start, end)\n        dist, prev, done = {start: 0}, {}, set()\n        heap = [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            if node in done:\n                continue\n            done.add(node)\n            if node == end:\n                break\n            for nxt, km in sorted(self.paths[node].items()):\n                if d + km < dist.get(nxt, float("inf")):\n                    dist[nxt] = d + km\n                    prev[nxt] = node\n                    heapq.heappush(heap, (d + km, nxt))\n        if end not in done:\n            raise NoRoute(f"no route from {start} to {end}")\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        stops.reverse()\n        return Route(stops, round(dist[end], 2), self.climb(stops))\n')
with open('waymark_map.py', 'w', encoding='utf-8') as f:
    f.write('from waymark_domain import PathNetwork\n\nHEIGHTS = {"Mill": 40, "Weir": 55, "Church": 80, "Ridge": 310, "Farm": 120, "Wood": 150, "Lake": 60, "Island": 5}\nPATHS = [("Mill", "Weir", 1.5), ("Weir", "Church", 2.0), ("Mill", "Church", 4.0), ("Church", "Ridge", 3.0),\n         ("Weir", "Farm", 2.5), ("Farm", "Ridge", 2.0), ("Farm", "Wood", 1.5), ("Wood", "Lake", 2.0), ("Lake", "Mill", 3.5)]\n\n\ndef sample_network():\n    network = PathNetwork(HEIGHTS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n')
with open('waymark_kinds.py', 'w', encoding='utf-8') as f:
    f.write('KINDS = {\n    frozenset({"Mill", "Church"}): "road",\n    frozenset({"Lake", "Mill"}): "road",\n    frozenset({"Weir", "Farm"}): "track",\n    frozenset({"Wood", "Lake"}): "track",\n}\n\n\ndef kind_of(a, b):\n    """The kind of the path between two junctions: road, track, or (by default) path."""\n    return KINDS.get(frozenset({a, b}), "path")\n')

**Predict before you run:** what will this print?



In [ ]:
from waymark_map import sample_network
from waymark_kinds import kind_of

network = sample_network()
route = network.shortest("Mill", "Ridge")
for a, b in zip(route.stops, route.stops[1:]):
    km = network.paths[a][b]
    climb = max(0, network.heights[b] - network.heights[a])
    print(f"{a:6} -> {b:6} {km} km  {kind_of(a, b):5}  climbs {climb} m ({round(climb / km)} m per km)")

> **Key idea.** Searches find the cheapest route for any non-negative cost, so express preferences as a cost per step (distance times factors for roads or steep uphill sections) and leave the search unchanged. Report real distances and times, not costs. Model preferences as a frozen dataclass that validates itself, parsed from the query string with clear errors. Explain why a route was chosen, and offer distinct alternatives with merged, honest labels.

### Exercise: Preferences as a value

Write a frozen dataclass `Preferences` with fields `avoid_roads=False`, `avoid_steep=False`, `road_factor=3.0`, `steep_factor=2.0` and `steep_limit=80`. In `__post_init__`, raise `ValueError("factors must be at least 1")` if either factor is below 1, and `ValueError("steep_limit must be positive")` unless it's above 0. Add a class method `from_query(query)` for a dict of strings: `avoid` is a comma-separated list of `roads` and `steep` (anything else: `ValueError("cannot avoid X")`); `steep_limit` is a whole number (`ValueError("steep_limit must be a whole number")`); other keys are ignored.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Preferences:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import dataclasses\n\ndef test_prefs():\n    \"valid, immutable, parsed\"\n    p = Preferences.from_query({\"avoid\": \"roads,steep\", \"steep_limit\": \"80\", \"from\": \"Mill\"})\n    assert p == Preferences(avoid_roads=True, avoid_steep=True, steep_limit=80)\n    assert Preferences.from_query({}) == Preferences() and Preferences.from_query({\"avoid\": \"\"}) == Preferences()\n    try:\n        p.avoid_roads = False\n    except dataclasses.FrozenInstanceError:\n        pass\n    else:\n        raise AssertionError(\"frozen\")\n    for query, message in (({\"avoid\": \"roads,cows\"}, \"cannot avoid cows\"), ({\"steep_limit\": \"steep\"}, \"steep_limit must be a whole number\"),\n                           ({\"steep_limit\": \"0\"}, \"steep_limit must be positive\")):\n        try:\n            Preferences.from_query(query)\n        except ValueError as e:\n            assert str(e) == message, str(e)\n        else:\n            raise AssertionError(query)\n    try:\n        Preferences(road_factor=0.5)\n    except ValueError as e:\n        assert str(e) == \"factors must be at least 1\"\n    assert hash(p) == hash(Preferences(avoid_roads=True, avoid_steep=True, steep_limit=80))")

### Exercise: The cost of a step

With the page's `kind_of`, write `step_cost(network, a, b, prefs)`: the path's length, times `prefs.road_factor` if `prefs.avoid_roads` and the path is a road, times `prefs.steep_factor` if `prefs.avoid_steep` and the step from `a` to `b` climbs more than `prefs.steep_limit` metres per km (uphill only). Use the given `Preferences`.

In [ ]:
from dataclasses import dataclass
from waymark_kinds import kind_of
from waymark_map import sample_network

@dataclass(frozen=True)
class Preferences:
    avoid_roads: bool = False
    avoid_steep: bool = False
    road_factor: float = 3.0
    steep_factor: float = 2.0
    steep_limit: int = 80

def step_cost(network, a, b, prefs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cost():\n    \"per step, per direction\"\n    net = sample_network()\n    plain, both = Preferences(), Preferences(avoid_roads=True, avoid_steep=True)\n    assert step_cost(net, \"Farm\", \"Ridge\", plain) == 2.0\n    assert step_cost(net, \"Farm\", \"Ridge\", both) == 4.0 and step_cost(net, \"Ridge\", \"Farm\", both) == 2.0, \"downhill isn't steep\"\n    assert step_cost(net, \"Mill\", \"Church\", both) == 12.0 and step_cost(net, \"Lake\", \"Mill\", Preferences(avoid_roads=True, road_factor=1.5)) == 5.25\n    assert step_cost(net, \"Weir\", \"Farm\", both) == 2.5")

### Exercise: The cheapest route

Write `best_route(network, start, end, prefs)` using the given `step_cost`: Dijkstra on cost (neighbours in name order). Return `(route, cost)`: a domain `Route` with the **real** km (rounded to 2 places) and climb, and the total cost rounded to 2 places. Raise `UnknownPlace` and `NoRoute` as the domain does.

In [ ]:
import heapq
from dataclasses import dataclass
from waymark_domain import NoRoute, Route, UnknownPlace
from waymark_kinds import kind_of
from waymark_map import sample_network

@dataclass(frozen=True)
class Preferences:
    avoid_roads: bool = False
    avoid_steep: bool = False
    road_factor: float = 3.0
    steep_factor: float = 2.0
    steep_limit: int = 80

def step_cost(network, a, b, prefs):
    km = network.paths[a][b]
    cost = km * (prefs.road_factor if prefs.avoid_roads and kind_of(a, b) == "road" else 1)
    climb = max(0, network.heights[b] - network.heights[a])
    return cost * (prefs.steep_factor if prefs.avoid_steep and climb / km > prefs.steep_limit else 1)

def best_route(network, start, end, prefs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_best():\n    \"preferences change the route, not the search\"\n    net = sample_network()\n    route, cost = best_route(net, \"Mill\", \"Ridge\", Preferences())\n    assert route.stops == [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"] and cost == 6.0\n    route, cost = best_route(net, \"Mill\", \"Ridge\", Preferences(avoid_steep=True))\n    assert route.stops == [\"Mill\", \"Weir\", \"Church\", \"Ridge\"] and (route.km, cost) == (6.5, 6.5)\n    route, cost = best_route(net, \"Mill\", \"Wood\", Preferences(avoid_roads=True))\n    assert route.stops == [\"Mill\", \"Weir\", \"Farm\", \"Wood\"] and (route.km, route.climb_m, cost) == (5.5, 110, 5.5)\n    for args, kind in (((\"Mill\", \"Island\"), NoRoute), ((\"Moon\", \"Mill\"), UnknownPlace)):\n        try:\n            best_route(net, *args, Preferences())\n        except kind:\n            pass\n        else:\n            raise AssertionError(args)")

### Exercise: Explaining a route

Write `explain(network, stops, shortest_km, steep_limit=80)`: a list of plain sentences about a route, in this order: `"X km on roads"` (if any; X rounded to 1 place); `"1 steep climb: Farm to Ridge"` or `"2 steep climbs: A to B, C to D"` (steps climbing more than `steep_limit` m per km, in route order), or `"no steep climbs"`; and, if the route is more than 0.05 km longer than `shortest_km`, `"X km longer than the shortest route"` (1 place).

In [ ]:
from waymark_kinds import kind_of
from waymark_map import sample_network

def explain(network, stops, shortest_km, steep_limit=80):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_explain():\n    \"reasons in plain words\"\n    net = sample_network()\n    assert explain(net, [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"], 6.0) == [\"1 steep climb: Farm to Ridge\"]\n    assert explain(net, [\"Mill\", \"Weir\", \"Church\", \"Ridge\"], 6.0) == [\"no steep climbs\", \"0.5 km longer than the shortest route\"]\n    assert explain(net, [\"Mill\", \"Church\", \"Weir\", \"Farm\", \"Ridge\"], 6.0) == [\"4.0 km on roads\", \"1 steep climb: Farm to Ridge\", \"4.5 km longer than the shortest route\"]\n    assert explain(net, [\"Ridge\", \"Farm\", \"Weir\", \"Mill\"], 6.0) == [\"no steep climbs\"]\n    assert explain(net, [\"Lake\", \"Wood\", \"Farm\", \"Ridge\"], 5.5, steep_limit=40) == [\"2 steep climbs: Lake to Wood, Farm to Ridge\"]")

### Bug hunt: Downhill counted as steep

The steepness check uses the size of the height change, so walking **down** from the ridge counts as a steep climb: it prints `steep: True`. Count only climbing, so it prints `steep: False`.

In [ ]:
heights = {"Ridge": 310, "Farm": 120}
km = 2.0
rate = abs(heights["Farm"] - heights["Ridge"]) / km
print("steep:", rate > 80)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"uphill only\"\n    expect_output(\"steep: False\")")

### Bug hunt: A list as a dataclass default

Dataclasses refuse a mutable default (one list shared by every instance), so this crashes with `ValueError: mutable default <class 'list'> for field avoid is not allowed`. Use `field(default_factory=list)`, so it prints `[] ['roads']`.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Preferences:
    avoid: list = []

a, b = Preferences(), Preferences()
b.avoid.append("roads")
print(a.avoid, b.avoid)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a fresh list each time\"\n    expect_output(\"[] ['roads']\")")

### Puzzle: Preferences through the layers

Arrange the API handler: parse the preferences (a bad one is a 400), find the best route, and explain it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the cost of a 2 km road for a walker avoiding roads with a road factor of 3
q = ___   # True or False: Dijkstra's algorithm works with any non-negative costs
r = ___   # metres per km for a climb from 80 m to 310 m over 3 km (rounded to a whole number)
s = ___   # the exception a frozen dataclass raises on assignment (its class name)
t = ___   # the HTTP status for ?avoid=cows

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 6 and q is True and r == 77, \"p, q and r\"\n    assert s == \"FrozenInstanceError\" and t == 400, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: three honest alternatives

Write `alternatives(network, start, end)` using the given helpers. Find the best route for three labelled preferences, in this order: `"shortest"` (`Preferences()`), `"flattest"` (avoiding steep climbs) and `"quietest"` (avoiding roads). Merge routes with the same stops into one entry whose label joins the labels in that order with `" and "`. Return a list of dicts `{"label", "stops", "km", "minutes", "why"}`, one per distinct route, in the order their first label appears, where `why` is `explain(...)` against the shortest route's km.

In [ ]:
import heapq
from dataclasses import dataclass
from waymark_domain import NoRoute, Route, UnknownPlace
from waymark_kinds import kind_of
from waymark_map import sample_network

@dataclass(frozen=True)
class Preferences:
    avoid_roads: bool = False
    avoid_steep: bool = False
    road_factor: float = 3.0
    steep_factor: float = 2.0
    steep_limit: int = 80

def step_cost(network, a, b, prefs):
    km = network.paths[a][b]
    cost = km * (prefs.road_factor if prefs.avoid_roads and kind_of(a, b) == "road" else 1)
    climb = max(0, network.heights[b] - network.heights[a])
    return cost * (prefs.steep_factor if prefs.avoid_steep and climb / km > prefs.steep_limit else 1)

def best_route(network, start, end, prefs):
    best, prev, done, heap = {start: 0}, {}, set(), [(0, start)]
    while heap:
        c, node = heapq.heappop(heap)
        if node in done:
            continue
        done.add(node)
        if node == end:
            stops = [end]
            while stops[-1] != start:
                stops.append(prev[stops[-1]])
            stops.reverse()
            km = sum(network.paths[a][b] for a, b in zip(stops, stops[1:]))
            return Route(stops, round(km, 2), network.climb(stops)), round(c, 2)
        for nxt in sorted(network.paths[node]):
            step = step_cost(network, node, nxt, prefs)
            if c + step < best.get(nxt, float("inf")):
                best[nxt], prev[nxt] = c + step, node
                heapq.heappush(heap, (c + step, nxt))
    raise NoRoute(f"no route from {start} to {end}")

def explain(network, stops, shortest_km, steep_limit=80):
    steps = list(zip(stops, stops[1:]))
    out = []
    road = sum(network.paths[a][b] for a, b in steps if kind_of(a, b) == "road")
    if road:
        out.append(f"{round(road, 1)} km on roads")
    steep = [f"{a} to {b}" for a, b in steps if max(0, network.heights[b] - network.heights[a]) / network.paths[a][b] > steep_limit]
    out.append(f"{len(steep)} steep climb{'s' if len(steep) > 1 else ''}: {', '.join(steep)}" if steep else "no steep climbs")
    km = sum(network.paths[a][b] for a, b in steps)
    if km - shortest_km > 0.05:
        out.append(f"{round(km - shortest_km, 1)} km longer than the shortest route")
    return out

def alternatives(network, start, end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_alternatives():\n    \"distinct routes, honest labels\"\n    net = sample_network()\n    alts = alternatives(net, \"Mill\", \"Ridge\")\n    assert [(a[\"label\"], a[\"stops\"], a[\"km\"], a[\"minutes\"]) for a in alts] == [\n        (\"shortest and quietest\", [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"], 6.0, 99),\n        (\"flattest\", [\"Mill\", \"Weir\", \"Church\", \"Ridge\"], 6.5, 105)]\n    assert alts[1][\"why\"] == [\"no steep climbs\", \"0.5 km longer than the shortest route\"]\n    one = alternatives(net, \"Ridge\", \"Mill\")\n    assert len(one) == 1 and one[0][\"label\"] == \"shortest and flattest and quietest\"\n    wood = alternatives(net, \"Mill\", \"Wood\")\n    assert [a[\"label\"] for a in wood] == [\"shortest and flattest\", \"quietest\"] and wood[1][\"stops\"] == [\"Mill\", \"Weir\", \"Farm\", \"Wood\"]")

## Warm-up

The shortest route to Ridge finishes with a climb of 190 m in 2 km. For a walker who wants to avoid steep sections, a slightly longer route might be better; for one who wants to keep off roads, this one is fine. The map file of Week 88 gains a kind for each path; open `waymark_kinds.py` above to see the sample's.

## Cost, not just distance

Dijkstra's algorithm never cared that the numbers on the paths were kilometres: it finds the route with the smallest **sum of costs**, for any costs that aren't negative. So preferences become a **cost function**: for each step from `a` to `b`,

- start with the distance;
- multiply by a **road factor** (say 3) if the walker avoids roads and the path is a road: a road counts as three times as long, so it's used only when the alternative is more than three times longer;
- multiply by a **steep factor** (say 2) if the walker avoids steep climbs and this step climbs more than the **steep limit** (say 80 m per km). Only uphill counts: the cost is per direction.

The search returns the cheapest route; the **reported** distance and time are still the real ones. Keeping the search generic and putting the policy in one cost function means a new preference (avoid fields with cattle, prefer shade) changes one function, not the algorithm.

## Preferences as a value

Preferences arrive as a query string (`?avoid=roads,steep&steep_limit=80`) and travel through the service to the search. Model them as a **frozen dataclass** (Week 32): immutable (so a cached result keyed by them can't be corrupted), comparable, printable, and **validating itself** in `__post_init__`, so an invalid set of preferences can't exist. Parse the query string in a class method, `Preferences.from_query(...)`, which raises `ValueError` with a clear message for anything unknown: an API answers that with `400`.

## The cheapest route

The search is Dijkstra with `cost(a, b)` in place of `paths[a][b]`. Track the real distance alongside the cost (or recompute it from the stops) so the route reports kilometres, not cost units. One subtlety: costs must never be negative, which multiplication by factors of at least 1 guarantees, and which the preferences' validation enforces.

## Explaining and offering choices

A route that's longer than the obvious one needs a reason, or users assume it's a bug. **Explain** the route in plain words: how far on roads, how many steep sections and where, and how much longer it is than the shortest. Better still, **offer** a few alternatives (the shortest, the flattest, the quietest), each labelled, and merge identical ones: if the shortest route is also the quietest, show it once as "shortest and quietest" rather than three identical lines.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Options as values**: find a place where your capstone passes several loose settings around (flags, a dict of options) and make it a frozen dataclass that validates itself.
2. **Policy in one place**: if a rule could change (a price, a ranking, a threshold), put it in one function and test it on its own.
3. **Explain one decision** your program makes, in plain words, in its output.
4. **Merge** and update the burndown. The build days are over: tomorrow is tests.

## Before you go

Check your burndown: what will be finished by Friday, and what carries over? Decide now. Tomorrow is the **tests** day for sprint 2: performance budgets, a matrix of preferences, and a regression suite for the search.